# GPA778 - Lab 2
# Part A - Perception I: Image Processing and Lidar

**Objective:** Develop the `jerro_perception` package to integrate the robot's perception systems (2D camera and Lidar) and create a launch file to start all nodes.

## Laboratory Objectives

In this lab, you will develop two ROS 2 nodes for the **jerro_perception** package:

| Node | Description | Milestone |
|------|-------------|-----------|
| `cam_traitement` | Applies an edge detection filter (Canny) to camera images | Milestone 1 |
| `lidar_traitement` | Converts LaserScan data into Cartesian coordinates and publishes them as PointCloud2 | Milestone 2 |

You will also create a **launch file** to start all nodes with a single command (Milestone 3).

### What you will learn:
- Create a ROS 2 Python package
- Use **cv_bridge** to convert ROS messages into OpenCV images
- Apply the Canny filter for edge detection
- Process Lidar data (polar → Cartesian conversion)
- Write a ROS 2 launch file

## Filtering an Image from the Camera

### Configuring the Camera

Jerro has a camera attached to its front, connected via CSI to the Raspberry Pi. To obtain images on a ROS 2 topic, we use the [`v4l2_camera`](https://docs.ros.org/en/rolling/p/v4l2_camera/) package:
```bash
ros2 run v4l2_camera v4l2_camera_node --ros-args -p image_raw.enable_pub_plugins:="['image_transport/raw','image_transport/compressed']"
# The displayed errors are "normal"
```

This package connects to the camera data stream and publishes it to the following topics:
```bash
ubuntu@ubuntu:~/ros2_ws$ ros2 topic list
/camera_info  # The camera calibration parameters
/image_raw    # The image directly coming from the camera
```

For more information about these messages: [Image](https://docs.ros.org/en/noetic/api/sensor_msgs/html/msg/Image.html), [CameraInfo](https://docs.ros.org/en/melodic/api/sensor_msgs/html/msg/CameraInfo.html).

### Calibration File

By listening to the `/camera_info` topic, you will notice that all parameters are set to 0. This stems from the following error when starting the node:
```bash
[ERROR] [camera_calibration_parsers]: Unable to open camera calibration file [/home/ubuntu/.ros/camera_info/mmal_service_16.1.yaml]
```

To fix this error, copy the provided `mmal_service_16.1.yaml` file from Moodle into the `/home/ubuntu/.ros/camera_info/` directory. After restarting the node, the `/camera_info` topic should contain the calibration values.

### Package Creation

In your workspace, create the `jerro_perception` package with the required dependencies:
```bash
ros2 pkg create --build-type ament_python jerro_perception \
    --dependencies rclpy sensor_msgs std_msgs cv_bridge sensor_msgs_py
```

### Conversion to OpenCV

The `cv_bridge` library allows converting ROS 2 images to [OpenCV](https://docs.opencv.org/4.x/d6/d00/tutorial_py_root.html) images (and vice versa). OpenCV is a library offering numerous image processing operations.

To import the library:
```python
from cv_bridge import CvBridge
```

In the node initialization:
```python
self.bridge = CvBridge()
```

To convert images:
```python
# ROS 2 message to OpenCV
cv_image = self.bridge.imgmsg_to_cv2(msg, "bgr8")

# OpenCV to ROS 2 message
ros_image = self.bridge.cv2_to_imgmsg(cv_image, encoding="mono8")
```

### The Canny Filter

The Canny filter is an edge detection algorithm. To apply it, import OpenCV:
```python
import cv2
```



```python
# 2. Apply Canny filter
img_canny = cv2.Canny(cv_image, t_lower, t_upper)
```

The parameters `t_lower` and `t_upper` control the detection thresholds of the filter (e.g., 50 and 150).

## Using the Lidar

### Installing the sllidar_ros2 Package

Jerro's Lidar is a SLAMTEC S2. To obtain data on a ROS 2 topic, you need to install the `sllidar_ros2` package:
```bash
cd ~/ros2_ws/src
git clone [https://github.com/Slamtec/sllidar_ros2.git](https://github.com/Slamtec/sllidar_ros2.git)
cd ~/ros2_ws
colcon build --packages-select sllidar_ros2
source install/setup.bash
```

To launch the Lidar driver:
```bash
ros2 launch sllidar_ros2 sllidar_s2_launch.py
```

The driver publishes data on the `/scan` topic using the `sensor_msgs/msg/LaserScan` type. For more information about this message: [LaserScan](https://docs.ros2.org/latest/api/sensor_msgs/msg/LaserScan.html).

### Converting LaserScan Data

The LaserScan message contains measured distances in polar coordinates (angle + distance). To convert them into Cartesian coordinates (x, y, z), a function is provided on Moodle: `laser_scan_to_array.py`.

Copy this file into your package:
```bash
cp laser_scan_to_array.py ~/ros2_ws/src/jerro_perception/jerro_perception/
```

To use this function in your node:
```python
from jerro_perception.laser_scan_to_array import laser_scan_to_array

# In the subscriber callback
points = laser_scan_to_array(msg, remove_invalid_ranges=True)
# points is a NumPy array of shape (N, 3) with columns [x, y, z]
```

The `remove_invalid_ranges=True` parameter filters out invalid values (`inf`, `nan`) that may appear when the Lidar does not detect an obstacle.

### Publishing to PointCloud2

To publish the converted points, use the `sensor_msgs/msg/PointCloud2` message. The `sensor_msgs_py` library provides utility functions to create this message:

```python
from sensor_msgs.msg import PointCloud2
from sensor_msgs_py import point_cloud2

# In the callback, after conversion
points = laser_scan_to_array(msg, remove_invalid_ranges=True)

# Create the PointCloud2 message
cloud_msg = point_cloud2.create_cloud_xyz32(msg.header, points)

# Publish to /points
self.publisher.publish(cloud_msg)
```

`msg.header` is reused to preserve the timestamp and frame_id from the original message.

## Creating a Launch File

A launch file allows you to start multiple nodes with a single command. In ROS 2, launch files are written in Python.

### Launch Directory Structure

Create a `launch` folder in your package:
```bash
mkdir ~/ros2_ws/src/jerro_perception/launch
```

### Writing the Launch File

Create the file `perception_launch.py` inside the `launch` folder. Here is the basic structure of a ROS 2 launch file:

```python
from launch import LaunchDescription
from launch_ros.actions import Node

def generate_launch_description():
    return LaunchDescription([
        # Add your nodes here
    ])
```

### Adding Nodes with Parameters

To add a node to the launch file, use the `Node` class:
```python
Node(
    package='package_name',
    executable='executable_name',
    name='node_name',
    parameters=[{
        'param1': value1,
        'param2': value2
    }]
)
```

#### Example: v4l2_camera Node with Parameters

The camera node requires a special parameter. Recall that we previously ran it with:
```bash
ros2 run v4l2_camera v4l2_camera_node --ros-args -p image_raw.enable_pub_plugins:="['image_transport/raw','image_transport/compressed']"
```

To translate this command into a launch file, the parameter name remains the same (with the dot) and the value is a list of strings:

```python
Node(
    package='v4l2_camera',
    executable='v4l2_camera_node',
    name='v4l2_camera_node',
    parameters=[{
        'image_raw.enable_pub_plugins': ['image_transport/raw','image_transport/compressed']
    }]
)
```

**Note:** Parameters with dots in their names (like `image_raw.enable_pub_plugins`) are valid in ROS 2 and must be written exactly as they appear in the original command.

**To do:** Add the 3 required nodes to your launch file:
1. The `v4l2_camera_node` node from the `v4l2_camera` package (with the parameter above)
2. The `cam_traitement` node from your package
3. The `lidar_traitement` node from your package

### Including an Existing Launch File

To include a launch file from another package (such as the Lidar one), use `IncludeLaunchDescription`. This lets you reuse existing launch files without duplicating their contents.

**Additional required imports:**
```python
from launch.actions import IncludeLaunchDescription
from launch.launch_description_sources import PythonLaunchDescriptionSource
from ament_index_python.packages import get_package_share_directory
```

**Example to include the Lidar launch file:**
```python
IncludeLaunchDescription(
    PythonLaunchDescriptionSource([
        get_package_share_directory('sllidar_ros2'),
        '/launch/sllidar_s2_launch.py'
    ])
)
```

The `get_package_share_directory()` function returns the path to the installed package's `share` folder. This allows locating the launch file regardless of where the workspace is installed.

**To do:** Include the Lidar launch file in your `perception_launch.py` so that everything starts with a single command.

### Configuring setup.py

For the launch file to be installed with the package, you must update the `setup.py` file.

**1. Add the imports at the top of the file:**
```python
import os
from glob import glob
```

**2. Inside the `data_files` list, add this line:**
```python
(os.path.join('share', package_name, 'launch'), glob('launch/*.py')),
```

**3. Inside `entry_points`, add your two nodes:**
```python
entry_points={
    'console_scripts': [
        'cam_traitement = jerro_perception.cam_traitement:main',
        'lidar_traitement = jerro_perception.lidar_traitement:main',
    ],
},
```

### Launching

After compiling the package, run the launch file using:
```bash
cd ~/ros2_ws
colcon build --packages-select jerro_perception
source install/setup.bash
ros2 launch jerro_perception perception_launch.py
```

All nodes should start simultaneously. You can verify this with:
```bash
ros2 node list
```

## Verification

Use the following commands to check if your nodes are running correctly:

```bash
# List active topics
ros2 topic list

# Verify that /image_canny is being published
ros2 topic hz /image_canny

# Verify that /points is being published
ros2 topic hz /points

# List active nodes
ros2 node list
```

**Visualization with Rviz:**
1. Connect to the robot via Rviz Studio
2. Add an **Image** display panel to visualize `/image_raw` and `/image_canny` side-by-side
3. Add a **PointCloud2** display panel to visualize the `/points` point cloud

## Lab Instructor Demonstration

At the end of this lab, you must demonstrate:

| Milestone | Expected Demonstration |
|-----------|------------------------|
| 1 | The `cam_traitement` node publishes detected edges to `/image_canny` |
| 2 | The `lidar_traitement` node publishes converted points to `/points` |
| 3 | The launch file starts all nodes with a single command |